# <center>DuckDB on Kaggle: SQL Analytics Without a Database</center>

<center>

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python&logoColor=white)
![DuckDB](https://img.shields.io/badge/DuckDB-1.x-FFF000?logo=duckdb&logoColor=black)
![pandas](https://img.shields.io/badge/pandas-2.x-150458?logo=pandas)
![Parquet](https://img.shields.io/badge/Apache-Parquet-50ABF1)
![License](https://img.shields.io/badge/License-MIT-red)

</center>

---

**Author:** Lorenzo Scaturchio  
**Last Updated:** July 2026  
**Kernel Version:** 1.0

---

## TL;DR

DuckDB is an **in-process SQL engine** — SQLite's "just `import` it, no server"
model, but built column-first for analytics. The honest headline this notebook
earns with live benchmarks: DuckDB is **not** a faster pandas. It is a SQL
analytics engine you point at *files*, and it wins decisively — **6-16x in our
runs** — on the two things pandas is worst at: scanning Parquet too big to load,
and window functions. On in-memory dataframe wrangling that pandas already does
well — plain group-bys and joins — it does not help, and can be slower. Two of
our four benchmarks favour pandas. Knowing *which* is which is the whole skill,
and that is what we measure below.

## Table of Contents

1. [Objective](#1.-Objective)
2. [What DuckDB Is (and Is Not)](#2.-What-DuckDB-Is-(and-Is-Not))
3. [Querying DataFrames in Place](#3.-Querying-DataFrames-in-Place)
4. [The Superpower: Query Parquet Without Loading It](#4.-The-Superpower)
5. [Benchmark Method](#5.-Benchmark-Method)
6. [Benchmarks: Group-by, Parquet Scan, Window, Join](#6.-Benchmarks)
7. [Results & Interpretation](#7.-Results-&-Interpretation)
8. [Larger-than-RAM: Out-of-Core Aggregation](#8.-Larger-than-RAM)
9. [Interop: pandas, Polars, Arrow](#9.-Interop)
10. [SQL Patterns Cheatsheet](#10.-SQL-Patterns-Cheatsheet)
11. [Conclusion & Next Experiments](#11.-Conclusion)

## 1. Objective

Every Kaggle competition with more than a few hundred MB of data eventually
forces the same question: *do I really have to load this whole file into a
pandas DataFrame just to compute a few aggregates?* With DuckDB the answer is
no — you write SQL against the file on disk and it reads only what the query
touches.

By the end of this notebook you will be able to:

- run SQL directly against **pandas DataFrames** and **Parquet/CSV files** with
  zero setup (`import duckdb`, that is the entire install story);
- recognise the workloads where DuckDB gives a 10x speedup and the ones where
  it gives nothing, from measured evidence rather than hype;
- aggregate a table **larger than kernel RAM** without ever loading it whole;
- move results between DuckDB, pandas, Polars, and Arrow at near-zero cost.

Everything runs on the standard Kaggle CPU kernel — DuckDB has no server, no
daemon, and no configuration.

In [ ]:
%pip install -q -U duckdb pyarrow

import time
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print(f"duckdb {duckdb.__version__} | pandas {pd.__version__} | numpy {np.__version__}")

## 2. What DuckDB Is (and Is Not)

| | SQLite | pandas | **DuckDB** |
|---|---|---|---|
| Runs in your process, no server | yes | yes | **yes** |
| Storage orientation | row | column (in RAM) | **column** |
| Built for analytical queries (scan-heavy aggregates) | no | partly | **yes** |
| Queries files on disk without loading them | no | no | **yes** |
| Interface | SQL | Python API | **SQL** |

The one-sentence mental model: **DuckDB is to analytics what SQLite is to
transactions** — an embedded engine you `import`, not a database you run. The
column-first storage is why it beats SQLite on the group-by/scan workloads that
define competition feature engineering.

We build a **5,000,000-row synthetic e-commerce transactions table** with a
fixed seed and write it once to Parquet. The Parquet file is what makes the
out-of-core benchmarks below honest: DuckDB will query it directly from disk.

In [ ]:
N = 5_000_000

pdf = pd.DataFrame({
    "user_id":    rng.integers(1, 300_001, N),
    "product_id": rng.integers(1, 8_001, N),
    "category":   rng.choice(
        ["electronics", "fashion", "home", "sports", "beauty", "toys", "books", "grocery"], N),
    "country":    rng.choice(
        ["US", "GB", "DE", "FR", "JP", "BR", "IN", "CA", "AU", "IT", "ES", "MX"], N),
    "price":      rng.gamma(2.0, 25.0, N).round(2),
    "quantity":   rng.integers(1, 6, N),
    "ts":         pd.to_datetime("2024-07-01") + pd.to_timedelta(rng.integers(0, 730 * 24 * 3600, N), unit="s"),
})

PARQUET = "transactions.parquet"
pdf.to_parquet(PARQUET)

import os
print(f"rows: {len(pdf):,} | pandas RAM: {pdf.memory_usage(deep=True).sum() / 1e6:,.0f} MB "
      f"| parquet on disk: {os.path.getsize(PARQUET) / 1e6:,.1f} MB")

Note the compression: the same data is ~320 MB in pandas memory but under
100 MB as Parquet, because Parquet stores each column in a compact, typed,
compressed block. DuckDB exploits exactly that layout to skip past columns and
row-groups a query does not need.

## 3. Querying DataFrames in Place

The gateway drug: any pandas DataFrame in scope is queryable by name, with no
copy and no registration. DuckDB sees `pdf` as a table.

In [ ]:
result = duckdb.sql("""
    SELECT category,
           COUNT(*)                    AS n_orders,
           ROUND(AVG(price), 2)        AS avg_price,
           ROUND(SUM(price * quantity), 0) AS revenue
    FROM pdf
    GROUP BY category
    ORDER BY revenue DESC
""").df()

result

That returned a plain pandas DataFrame (`.df()`), so it drops straight back into
any downstream pandas/sklearn code. You can also chain SQL — the result of one
`duckdb.sql(...)` is itself a relation you can query again — which makes
multi-step feature pipelines read top-to-bottom instead of nesting.

## 4. The Superpower: Query Parquet Without Loading It

This is the feature that has no pandas equivalent. `read_parquet` inside a query
lets DuckDB apply **projection pushdown** (read only the columns in the SELECT)
and **predicate/row-group pruning** (skip blocks that cannot match the WHERE) —
so a filter-and-aggregate over one country touches a small fraction of the file,
and the other columns are never read off disk at all.

In [ ]:
us_by_category = duckdb.sql(f"""
    SELECT category, ROUND(SUM(price * quantity), 0) AS us_revenue
    FROM read_parquet('{PARQUET}')
    WHERE country = 'US'
    GROUP BY category
    ORDER BY us_revenue DESC
""").df()

us_by_category

The query named 3 of the table's 7 columns, so DuckDB read 3 columns off disk,
not 7 — and only the row-groups whose `country` statistics permit a `US` match.
You do not have to take that on faith: `EXPLAIN` prints the physical plan, and
the pushdown is visible in it.

In [ ]:
plan = duckdb.sql(f"""
    EXPLAIN SELECT category, SUM(price * quantity) AS us_revenue
    FROM read_parquet('{PARQUET}')
    WHERE country = 'US'
    GROUP BY category
""").fetchall()

print(plan[0][1])

Read the plan from the bottom up: the `PARQUET_SCAN` node lists only the
columns the query needs, with the `country=US` filter attached directly to the
scan — the filter runs *while reading the file*, not on a loaded table
afterwards. That placement is the entire out-of-core story, and on a real
competition file of tens of GB it is the difference between an instant answer
and an out-of-memory kernel crash. We quantify it next.

## 5. Benchmark Method

Same honest rules as any fair comparison:

- **Best of 3 runs** per operation (`time.perf_counter`).
- Both engines produce the **same result on the same data**; each cell asserts
  the row counts match.
- pandas gets its idiomatic vectorized form — no `.apply` strawmen.
- Every DuckDB timing **includes** materializing the result back to pandas with
  `.df()`, so the comparison is end-to-end, not SQL-only.
- Timings are recorded into one dict and the summary chart is drawn from them,
  so the picture reflects *this* kernel's hardware.

In [ ]:
con = duckdb.connect()  # an in-memory database; nothing is persisted
RESULTS = {}

def bench(label, fn, repeats=3):
    """Return fn() result; record best-of-N wall time under label."""
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - t0)
    RESULTS[label] = best
    print(f"{label:<30s} {best * 1000:>9.1f} ms")
    return out

## 6. Benchmarks

### 6.1 Simple in-memory group-by — *where DuckDB does NOT help*

Start with the honest loss. A plain group-by over an in-memory DataFrame is
something pandas is already good at, and DuckDB has to scan the DataFrame
through its Python interface to run the query.

In [ ]:
pd_gb = bench("groupby | pandas", lambda: (
    pdf.assign(rev=pdf.price * pdf.quantity)
       .groupby(["category", "country"], observed=True)
       .agg(rev_mean=("rev", "mean"), rev_sum=("rev", "sum"), n=("rev", "size"))
       .reset_index()
))

duck_gb = bench("groupby | duckdb", lambda: con.execute("""
    SELECT category, country, AVG(price*quantity) rev_mean,
           SUM(price*quantity) rev_sum, COUNT(*) n
    FROM pdf GROUP BY 1, 2
""").df())

assert len(pd_gb) == len(duck_gb)
print(f"speedup: {RESULTS['groupby | pandas'] / RESULTS['groupby | duckdb']:.1f}x  "
      f"(< 1.0 means pandas won — and that is fine)")

Expect this one to hover around parity or a slight pandas win. **That is the
point:** DuckDB is not magic pixie dust you sprinkle on in-memory pandas code.
Its wins come from the next three patterns.

### 6.2 Parquet scan + aggregate — *the out-of-core win*

Filter-and-aggregate straight off the Parquet file (DuckDB) versus loading the
file into pandas first and then aggregating.

In [ ]:
duck_pq = bench("parquet scan+agg | duckdb", lambda: con.execute(f"""
    SELECT category, SUM(price*quantity) rev
    FROM read_parquet('{PARQUET}')
    WHERE country = 'US' GROUP BY 1 ORDER BY 2 DESC
""").df())

pd_pq = bench("parquet read+agg | pandas", lambda: (
    (lambda d: d[d.country == "US"]
               .assign(rev=lambda x: x.price * x.quantity)
               .groupby("category", observed=True).rev.sum()
               .sort_values(ascending=False))(pd.read_parquet(PARQUET))
))

print(f"speedup: {RESULTS['parquet read+agg | pandas'] / RESULTS['parquet scan+agg | duckdb']:.1f}x")

### 6.3 Window function — *analytical SQL win*

Rank each user's transactions by price — the "top-N per group" pattern behind
recency/order features. `ROW_NUMBER() OVER (...)` in SQL versus a pandas
grouped rank.

In [ ]:
duck_win = bench("window rank | duckdb", lambda: con.execute("""
    SELECT user_id, price,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY price DESC) rn
    FROM pdf
""").df())

pd_win = bench("window rank | pandas", lambda: pdf.assign(
    rn=pdf.groupby("user_id").price.rank(method="first", ascending=False)
))

assert len(duck_win) == len(pd_win)
print(f"speedup: {RESULTS['window rank | pandas'] / RESULTS['window rank | duckdb']:.1f}x")

### 6.4 Analytical join — *the second honest loss*

We join a product dimension and roll up to margin-weighted profit per brand — a
small result, aggregated inside SQL. You might expect DuckDB to win, but when
both tables already live in memory as DataFrames this lands in pandas'
territory, right alongside the group-by: DuckDB must scan the wide in-memory
frame through its Python interface, and pandas' `merge` is hard to beat on data
it already holds.

In [ ]:
dim = pd.DataFrame({
    "product_id": np.arange(1, 8_001),
    "brand": rng.choice([f"brand_{i:03d}" for i in range(150)], 8_000),
    "margin": rng.uniform(0.05, 0.45, 8_000).round(3),
})

duck_join = bench("join+agg | duckdb", lambda: con.execute("""
    SELECT d.brand,
           ROUND(SUM(t.price * t.quantity * d.margin), 0) profit,
           COUNT(*) n
    FROM pdf t JOIN dim d USING (product_id)
    GROUP BY 1 ORDER BY 2 DESC
""").df())

pd_join = bench("join+agg | pandas", lambda: (
    pdf.merge(dim, on="product_id")
       .assign(p=lambda x: x.price * x.quantity * x.margin)
       .groupby("brand").agg(profit=("p", "sum"), n=("p", "size"))
       .sort_values("profit", ascending=False)
))

assert len(duck_join) == len(pd_join)
print(f"speedup: {RESULTS['join+agg | pandas'] / RESULTS['join+agg | duckdb']:.1f}x")

The lesson is not "DuckDB is bad at joins" — it is that a join between two
**already-in-memory** DataFrames is not where a SQL engine earns its keep. Flip
the same join to read from Parquet (`FROM read_parquet(...) t JOIN ...`) and
DuckDB's file-scan advantage returns, because now it is avoiding a load that
pandas cannot avoid. The rule holds: DuckDB wins when the query lets it skip
work, not when the data is already sitting in RAM.

## 7. Results & Interpretation

One chart from the timings above. Bars right of the dashed line are DuckDB
wins; a bar left of it means pandas was faster — and the chart shows both,
because an honest tool guide has to.

In [ ]:
pairs = [
    ("Group-by (in-memory)", "groupby"),
    ("Parquet scan+agg", "parquet"),
    ("Window rank", "window rank"),
    ("Join+agg", "join+agg"),
]

def pair_speedup(key):
    pandas_key = next(k for k in RESULTS if k.startswith(key) and "pandas" in k)
    duck_key   = next(k for k in RESULTS if k.startswith(key) and "duckdb" in k)
    return RESULTS[pandas_key] / RESULTS[duck_key]

labels = [p[0] for p in pairs]
speedups = [pair_speedup(p[1]) for p in pairs]
colors = ["#D64550" if s < 1 else "#2E7CD6" for s in speedups]

fig, ax = plt.subplots(figsize=(9, 3.6))
y = np.arange(len(labels))
bars = ax.barh(y, speedups, height=0.55, color=colors, zorder=3)
ax.bar_label(bars, labels=[f"{s:.1f}x" for s in speedups], padding=6, fontsize=11)
ax.axvline(1.0, color="#555555", linewidth=1, linestyle="--", zorder=2)
ax.text(1.02, -0.45, "pandas faster  <-  |  ->  DuckDB faster", color="#666666", fontsize=9)
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlabel("Speedup: pandas time / DuckDB time (log scale)")
ax.set_title(f"DuckDB vs pandas on {N/1e6:.0f}M rows — same kernel, best of 3", loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#DDDDDD", linewidth=0.6, zorder=0)
plt.tight_layout()
plt.show()

for lbl, s in zip(labels, speedups):
    verdict = "pandas wins" if s < 1 else "DuckDB wins"
    print(f"{lbl:<24s} {s:5.1f}x   {verdict}")

**Reading the chart:** two bars sit left of the line (in-memory group-by and
join — pandas territory) and two land far to the right (Parquet scan and
window). The unifying rule: **DuckDB wins when the query lets it avoid work** —
skipping unread Parquet columns and row-groups, or running a set-based window
in one pass. When the data is already a DataFrame in RAM and the operation is
something pandas is tuned for, the SQL round-trip is pure overhead. Underneath
both wins and losses sits one trade-off: SQL buys pushdown and streaming at the
cost of a serialization boundary with Python, so the more rows that must cross
that boundary, the faster the advantage erodes. That split — not a blanket "SQL
is faster" — is the takeaway. Fork this notebook and your bars will differ in
magnitude but not in shape.

## 8. Larger-than-RAM: Out-of-Core Aggregation

The reason DuckDB belongs in your Kaggle toolkit: it aggregates files that do
**not fit in memory**. Its execution engine spills to disk automatically, so a
query over a 50 GB Parquet file runs on a 13 GB kernel — something
`pd.read_parquet` simply cannot do (it would OOM on the load).

We can demonstrate the mechanism honestly on our file: query it **without ever
creating a DataFrame of it**, returning only the small aggregated result.

In [ ]:
# Whole pipeline in SQL: derive, filter, aggregate, rank — result is 12 rows.
monthly_top = con.execute(f"""
    WITH enriched AS (
        SELECT country,
               strftime(ts, '%Y-%m')       AS month,
               price * quantity             AS revenue
        FROM read_parquet('{PARQUET}')
    )
    SELECT country,
           ROUND(SUM(revenue), 0)                          AS total_revenue,
           ROUND(SUM(revenue) / COUNT(DISTINCT month), 0)  AS avg_monthly_revenue
    FROM enriched
    GROUP BY country
    ORDER BY total_revenue DESC
""").df()

print("Peak input was never materialised as a DataFrame; only these rows came back:")
monthly_top

Nothing in that cell held the 5M rows in Python at once — DuckDB streamed the
Parquet file through the aggregation and returned 12 rows. Swap the local path
for a glob like `read_parquet('data/*.parquet')` and the identical query spans
a whole partitioned dataset. For genuinely huge inputs, persist to a DuckDB
file (`duckdb.connect("comp.duckdb")`) so intermediate tables also spill to
disk instead of RAM.

## 9. Interop: pandas, Polars, Arrow

DuckDB is a good citizen of the dataframe ecosystem — it reads and returns
each format with a dedicated method, and the Arrow path is zero-copy. The
pragmatic Kaggle pattern: **build features in SQL, hand the small result to
whatever models it.**

In [ ]:
# SQL builds a user-level feature table; sklearn consumes a NumPy matrix.
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

features = con.execute("""
    SELECT user_id,
           COUNT(*)                     AS n_orders,
           ROUND(AVG(price), 3)         AS avg_price,
           SUM(quantity)                AS total_qty,
           COUNT(DISTINCT category)     AS n_categories,
           ROUND(SUM(price * quantity), 2) AS total_rev
    FROM pdf GROUP BY user_id
""").df()

X = features[["n_orders", "avg_price", "total_qty", "n_categories"]].to_numpy()
y = features["total_rev"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
model = Ridge(alpha=1.0).fit(X_tr, y_tr)

print(f"user features: {features.shape} | Ridge R^2: {r2_score(y_te, model.predict(X_te)):.3f}")

# Return formats: .df() -> pandas, .pl() -> polars, .arrow() -> Arrow (zero-copy)
try:
    import polars  # noqa: F401
    pl_out = con.execute("SELECT category, COUNT(*) n FROM pdf GROUP BY 1").pl()
    print(f".pl() returned a {type(pl_out).__module__.split('.')[0]} DataFrame: {pl_out.shape}")
except ImportError:
    print(".pl() would return a Polars DataFrame (polars not installed here)")

The high R² is expected — total revenue is mechanically tied to order count and
quantities, so this is a pipeline sanity check, not a modelling result. The
shape of the workflow is the lesson: one SQL statement turns 5M rows into a
300k-row feature table, and sklearn never sees SQL at all.

## 10. SQL Patterns Cheatsheet

The DuckDB idioms that cover most Kaggle work:

| Task | DuckDB |
|---|---|
| Query a DataFrame `df` | `duckdb.sql("SELECT * FROM df")` |
| Read Parquet lazily | `FROM read_parquet('file.parquet')` |
| Read many files | `FROM read_parquet('data/*.parquet')` |
| Read CSV (auto types) | `FROM read_csv_auto('file.csv')` |
| Result to pandas / Polars / Arrow | `.df()` / `.pl()` / `.arrow()` |
| Only some columns off disk | `SELECT a, b FROM read_parquet(...)` (pushdown) |
| Top-N per group | `ROW_NUMBER() OVER (PARTITION BY g ORDER BY x DESC)` |
| Rolling / lag features | `LAG(x) OVER (PARTITION BY g ORDER BY ts)` |
| Quantiles | `quantile_cont(x, 0.95)` |
| Pivot | `PIVOT tbl ON col USING SUM(val)` |
| Sample rows | `USING SAMPLE 10%` |
| Persist to disk (spill) | `duckdb.connect("comp.duckdb")` |

Ergonomics DuckDB adds on top of standard SQL that are worth knowing:

- **`SELECT * EXCLUDE (id, ts)`** — every column but a few, instead of typing
  the other forty.
- **`SELECT * REPLACE (price * 1.1 AS price)`** — transform one column, keep the
  rest untouched.
- **`GROUP BY ALL`** — group by every non-aggregated column automatically.
- **`COLUMNS('price_.*')`** — apply an aggregate across all columns matching a
  regex.

One caveat as you adopt these: they are DuckDB dialect, not portable SQL — fine
inside a kernel, worth flagging before pasting a query into a shared warehouse.
They are also easy to verify live, so let us run three of them on our table:

In [ ]:
print("-- GROUP BY ALL: no more repeating the SELECT columns --")
print(con.execute("""
    SELECT category, country, ROUND(SUM(price * quantity), 0) AS rev
    FROM pdf WHERE country IN ('US', 'GB')
    GROUP BY ALL ORDER BY rev DESC LIMIT 5
""").df().to_string(index=False))

print("\n-- EXCLUDE + REPLACE: tweak one column, keep the rest --")
print(con.execute("""
    SELECT * EXCLUDE (ts, user_id) REPLACE (ROUND(price * 1.1, 2) AS price)
    FROM pdf LIMIT 3
""").df().to_string(index=False))

print("\n-- COLUMNS() regex: one aggregate over every matching column --")
print(con.execute("""
    SELECT MAX(COLUMNS('(price|quantity)')) FROM pdf
""").df().to_string(index=False))

## 11. Conclusion

**Takeaways**

1. DuckDB is an in-process SQL analytics engine, not a faster pandas. Reach for
   it by **workload**, not by reflex.
2. In our measurements it won by **~6-16x on Parquet scans and window
   functions**, and lost on both in-memory dataframe ops we tried (group-by and
   join) — and the summary chart showed both directions honestly.
3. Its irreplaceable capability is **querying files too big to load**: only the
   needed columns and row-groups are read, and aggregates stream out-of-core.
4. Aggregate inside SQL and return a *small* result; do not materialise millions
   of joined rows back to Python and expect a speedup.

**Next steps — experiments to try on your own**

- Point `read_parquet` at a real multi-GB competition dataset and time a
   feature aggregation you currently do in chunked pandas. This is the
   recommended first experiment, because it is where the payoff is largest.
- Compare `.arrow()` versus `.df()` return time on a wide result — the zero-copy
   Arrow path can matter when the output is large.
- Rebuild one of your pandas feature pipelines as a single SQL CTE chain and
   diff the outputs to confirm they match.

**Related notebooks in this series:**

- Polars on Kaggle: The Complete Speed Guide
- Feature Engineering Cookbook: 50 Techniques
- Optuna Tuning: A Practical Kaggle Guide

---

**If this notebook helped you, please upvote!** Feedback and comments are very welcome.

*Lorenzo Scaturchio | July 2026*